# Test the L0 processor v1

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-609

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=4)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

In [ ]:
# Other imports
import glob
import shutil
import time
import os.path as osp

from datetime import timedelta
from IPython.display import JSON

from rs_common.prefect_utils import *

## Read the tasktables

Documentation: https://cpm.pages.eopf.copernicus.eu/eopf-cpm/main/processor-orchestration-guide/tasktables.html#tasktables

<div class="alert alert-block alert-warning">

Note: for now the L0 processor returns dummy values that are not usable.
</div>

In [ ]:
for process_id in "s1_l0", "s3_l0":
    tasktable: dict = dpr_client.get_process(process_id)
    print(f"Tasktable for {process_id!r}:")
    display(JSON(tasktable))
    # print(json.dumps(tasktable, indent=2))

## Run the L0 processor

<div class="alert alert-block alert-warning">

To be discussed: should the configuration and payload files for the processors be available from:

  * rs-dpr-service ?
  * rs-client-libraries ?
  * The user local disk ? (like in this demo)
</div>

In [ ]:
# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config_dir = osp.join(s3_base, "config")
s3_output_dir = osp.join(s3_base, "output")
s3_report_dir = osp.join(s3_base, "reports")

# Data properties
s1_short = {
    "process_name": "s1_l0", # process name in rs-dpr-service
    "payload_subpath": "s1/iw_joborder.short.yaml", # payload file path relative to the config dir
    "s3_output_dir": f"{s3_output_dir}/s1.short", # output dir in the s3 bucket
    "s3_report_dir": f"{s3_report_dir}/s1.short", # report dir in the s3 bucket
}
s1 = {
    "process_name": "s1_l0", 
    "payload_subpath": "s1/iw_joborder.yaml",
    "s3_output_dir": f"{s3_output_dir}/s1",
    "s3_report_dir": f"{s3_report_dir}/s1",
}
s3 = {
    "process_name": "s3_l0", 
    "payload_subpath": "s3/s3_dordop_payload.yaml",
    "s3_output_dir": f"{s3_output_dir}/s3",
    "s3_report_dir": f"{s3_report_dir}/s3",
}

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./config", s3_config_dir)

# Update local secret file depending on the environment, 
# and upload it again to the s3 bucket.
await dpr_client.update_configuration(
    local_path = "./config/secrets.json",
    s3_path = osp.join(s3_config_dir, "secrets.json"),
)

In [ ]:
# Number of dask workers
N_WORKERS = len(dask_client_eopf.scheduler_info()["workers"])

async def process_data(data: dict):
    """Process s1_short, s1 or s3 data"""

    # Remove existing output and report folders
    s3_delete(data["s3_output_dir"], log=True)
    s3_delete(data["s3_report_dir"], log=True)
    
    # Update local payload file depending on the environment, upload it to the s3 bucket,
    # and initialize output bucket folders.
    await dpr_client.update_configuration(
        local_path = osp.join("./config", data["payload_subpath"]),
        s3_path = osp.join(s3_config_dir, data["payload_subpath"]),
        is_payload = True,
        # Specific environment variables to expand in the payload file
        N_WORKERS=N_WORKERS,
        PREFECT_BUCKET_NAME=os.environ["PREFECT_BUCKET_NAME"], 
        OUTPUT_DIR=data["s3_output_dir"],
    )
    
    # Run processor
    start_time = time.time()
    result = dpr_client.run_process(
        data["process_name"],
        s3_config_dir = s3_config_dir,
        payload_subpath = data["payload_subpath"],
        s3_report_dir = data["s3_report_dir"],
    )
    try:
        dpr_client.wait_for_job(result, logger=logger, poll_interval=5)
    finally:
        print(f"Processor execution time: {str(timedelta(seconds=time.time() - start_time))}")
    
        # Download reports folder from the s3 bucket
        local_report_dir = f"./reports/{Path(data['s3_report_dir']).name}"
        shutil.rmtree(local_report_dir, ignore_errors=True)
        await s3_download_dir(data["s3_report_dir"], local_report_dir)
        
        # Display logs here
        local_log_file = glob.glob(osp.join(local_report_dir, "**/*.processor.log"), recursive=True)[0]
        with open(local_log_file, "r", encoding="utf-8") as openend:
            print(f"Log file {local_log_file!r}:\n{openend.read()}")

    # Download output data from the s3 bucket
    print(f"Output product generated on: {data['s3_output_dir']!r}")
    local_output_dir = f"./outputs/{Path(data['s3_output_dir']).name}"
    await s3_download_dir(data["s3_output_dir"], local_output_dir)
    print(f"Open it locally with QGis from: {osp.realpath(local_output_dir)!r}")

In [ ]:
# Run S1 short data
await process_data(s1_short)

16:04:45.261 | INFO    | prefect.S3Bucket - Delete from 'http://minio:9000': [
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/.empty",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/.zattrs",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/.zgroup",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/.zmetadata",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/.zattrs",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/.zgroup",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/.zmetadata",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/.zattrs",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/.zgroup",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/.zmetadata",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/conditions/.zattrs",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/conditions/.zgroup",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/conditions/calibration/.zattrs",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/conditions/calibration/.zgroup",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01SIWANC_20250611T050659_0008_A102_T502_76618_VH.zarr/conditions/calibration/application_process_identifier/.zarray",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/S01SIWRAW_20250611T050659_0008_A102_T667_76618_DV.zarr/S01SIWRAW_20250611T050659_0008_A102_T190_76618_VH.zarr/S01S

16:04:45.575 | INFO    | prefect.S3Bucket - Delete from 'http://minio:9000': [
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/reports/s1.short/logging_config.log",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/reports/s1.short/report.html"
]

16:04:45.587 [INFO] (rs_client.rs_client) Write empty file: 20 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/.empty'


16:04:45.599 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpmwj7il1f' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/.empty'.

16:04:45.611 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmp2_q1txi0' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

16:04:45.672 [INFO] (resources.utils) job_status: {'progress': 0, 'type': 'process', 'processID': 'dpr-service', 'created': '2025-08-01T16:04:45Z', 'started': '2025-08-01T16:04:45Z', 'updated': '2025-08-01T16:04:45Z', 'message': 'Sending task to the dask cluster', 'status': 'running', 'jobID': 'fac2c15a-2a10-4b54-8cc4-3663509d050e'}
16:04:45.673 [INFO] (resources.utils) -----  job 'fac2c15a-2a10-4b54-8cc4-3663509d050e': RUNNING 

16:04:50.691 [INFO] (resources.utils) job_status: {'progress': 50, 'type': 'process', 'processID': 'dpr-service', 'created': '2025-08-01T16:04:45Z', 'started': '2025-08-01T16:04:45Z', 'updated': '2025-08-01T16:04:45Z', 'message': 'In progress', 'status': 'running', 'jobID': 'fac2c15a-2a10-4b54-8cc4-3663509d050e'}
16:04:50.691 [INFO] (resources.utils) -----  job 'fac2c15a-2a10-4b54-8cc4-3663509d050e': RUNNING 

16:04:55.796 [INFO] (resources.utils) job_status: {'progress': 50, 'type': 'process', 'processID': 'dpr-service', 'created': '2025-08-01T16:04:45Z', 'st

In [ ]:
# Run S1 full data
if os.getenv("RSPY_FROM_CICD") != "1":
    await process_data(s1)

In [ ]:
# Run S3 full data
if os.getenv("RSPY_FROM_CICD") != "1":
    await process_data(s3)

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.